# Hash Table with Expiration (TTL)

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Onsite Loop, Caching, Hash Tables · **Difficulty/Frequency:** Rare (2/10)

> **See also:** [`24. Hash_Map`](../24.%20Hash_Map/24.%20Hash_Map.ipynb) — the hash table itself, without expiration. This problem is what happens when entries have a lifetime.

## Concepts

**What this problem is really testing:**
- Composing a **hash map** with a **min-heap** so two different questions are both fast
- **Lazy deletion** — and how a heap copes with entries it cannot reach in to change
- Knowing that **which clock you read** is a correctness decision, not a detail

**First-principles primer — what is each piece?**

- **TTL (time to live).** An entry has a deadline. Past it, the table must behave as though the key were never there — even if nothing has actively removed it yet.
- **Why not just check on read?** Checking `expiry <= now` inside `get` makes reads *correct*, but nothing ever reclaims a key nobody asks for again. A cache of a million short-lived keys leaks all million. You need a way to find expired entries **without scanning everything**.
- **Two questions, two structures.** Exactly the pairing from the [LRU cache](../10.%20LRU_Cache/10.%20LRU_Cache.ipynb), with a different second structure:

| Question | Structure | Cost |
|---|---|---|
| "What is the value for key K?" | hash map | O(1) |
| "What expires soonest?" | min-heap on expiry | O(1) to peek |

- **Lazy deletion.** A heap has no "find this entry and change it" operation. So when a key is overwritten or deleted, its old heap entry cannot be removed — it is left in place as **garbage**, and discarded when it eventually surfaces at the top. The map is the source of truth; the heap is a hint about *when to look*.

**The staleness test is the crux.** When `(expiry, key)` reaches the top of the heap, three things could be true:

| Situation | How you know | Action |
|---|---|---|
| The key was deleted | not in the map | pop, touch nothing |
| The key was overwritten | in the map, but with a **different** version | pop, touch nothing |
| Genuinely expired | in the map, versions match, `expiry <= now` | pop **and** delete from the map |
| Not expired yet | `expiry > now` | **stop** — everything behind it expires later |

That last row is what makes cleanup cheap: the heap is ordered, so the first non-expired entry means there is nothing more to do.

**Identify entries by a version, not by their timestamp.** The official answer tests `entry.expiry != heap_expiry` to detect an overwrite. Two entries can share an expiry — set a key, let it expire, set it again in the same tick with the same TTL — and then a **live** entry gets deleted. A monotonically increasing counter per `set` cannot collide, and costs one integer.

**And read the right clock.** `time.time()` is wall-clock: NTP corrections, daylight saving and manual changes all move it, and it can move **backwards**. Jump it back and every TTL silently extends; jump it forward and the whole table expires at once. `time.monotonic()` only ever increases. This is a correctness property, not a nicety.

**Simple worked example.** `set("a", 1, ttl=10)` at t=0, `set("b", 2, ttl=5)` at t=0, then at t=6:

```
heap:  [(5, "b"), (10, "a")]        map: {a: (1, exp 10), b: (2, exp 5)}

clean_expired() at t=6:
  top is (5, "b")  -> in the map, version matches, 5 <= 6  -> EXPIRED: pop and delete
  top is (10, "a") -> 10 > 6                               -> STOP
```

One pop, one comparison — not a scan of the table.

## Problem Statement

A hash table where every entry has a **time to live**:

| Method | Behaviour |
|---|---|
| `set(key, value, ttl)` | Store it; it becomes invisible after `ttl` seconds |
| `get(key)` | The value, or "missing" if absent **or expired** |
| `delete(key)` | Remove it now |
| `clean_expired()` | Reclaim everything past its deadline |
| `size()` | How many live entries |

Expiry is by **timeout**, not by least-recently-used.

### Approach 1 — Naive (a dict, scanned for cleanup)

**Idea:** store `(value, expiry)` in a dict. `get` checks the deadline; `clean_expired` walks every entry.

Reads are correct and O(1), which is what makes this tempting. The flaw is cleanup: finding expired keys means **looking at all of them**, including the 99% that are nowhere near their deadline.

**Time complexity:** O(1) `set`/`get`; **O(n) per cleanup**.

**Space complexity:** O(n).

In [ ]:
import heapq
import itertools
import time
from typing import Any, Callable, Dict, List, Optional, Tuple

_MISSING = object()          # a sentinel, so a stored None is not mistaken for "absent"


class NaiveExpiringTable:
    """Baseline: correct, but cleanup must scan every entry."""

    def __init__(self, clock: Callable[[], float] = time.monotonic) -> None:
        self._data: Dict[Any, Tuple[Any, float]] = {}
        self._clock = clock

    def set(self, key: Any, value: Any, ttl: float) -> None:
        self._data[key] = (value, self._clock() + ttl)

    def get(self, key: Any, default: Any = _MISSING) -> Any:
        entry = self._data.get(key)
        if entry is None:
            return default
        value, expiry = entry
        if expiry <= self._clock():
            del self._data[key]                  # expire on read
            return default
        return value

    def delete(self, key: Any) -> bool:
        return self._data.pop(key, _MISSING) is not _MISSING

    def clean_expired(self) -> int:
        now = self._clock()
        dead = [k for k, (_, exp) in self._data.items() if exp <= now]   # O(n) SCAN
        for k in dead:
            del self._data[k]
        return len(dead)

    def size(self) -> int:
        self.clean_expired()
        return len(self._data)

### Approach 2 — Optimal (hash map + min-heap, lazy deletion)

**Idea:** keep the map for lookup, and a min-heap of `(expiry, version, key)` so the soonest deadline is always visible in O(1).

**Three details that carry it:**

- **A version counter, not a timestamp, identifies an entry.** Every `set` takes the next integer from a counter. When a heap entry surfaces, it is live only if the map holds that **exact version**. Timestamps can repeat; a monotonic counter cannot. It also gives the heap a deterministic tie-break, so it never has to compare the keys themselves — which would fail outright for keys of mixed or non-comparable types.
- **`clean_expired` stops at the first live, unexpired entry.** Because the heap is ordered by expiry, everything behind it expires later. Cleanup costs only what it actually reclaims.
- **The clock is injected.** Tests advance a fake clock instead of sleeping, so a one-hour TTL is testable in microseconds — the same dependency-injection move as the `sleep` parameter in [Retry Strategy](../18.%20Retry_Strategy/18.%20Retry_Strategy.ipynb).

**On garbage.** A key overwritten a thousand times leaves 999 stale heap entries. They are harmless but not free, so `set` triggers a compaction once the heap grows well past the live set — otherwise a hot key's history is an unbounded leak.

**Time complexity:** **O(log n)** `set`, **O(1)** average `get`, **O(k log n)** cleanup for the k entries actually reclaimed.

**Space complexity:** O(n + garbage), bounded by the compaction threshold.

In [ ]:
class ExpiringHashTable:
    """Hash map for lookup + min-heap on expiry. Lazy deletion, monotonic clock."""

    def __init__(self, clock: Callable[[], float] = time.monotonic) -> None:
        self._data: Dict[Any, Tuple[Any, float, int]] = {}   # key -> (value, expiry, version)
        self._heap: List[Tuple[float, int, Any]] = []        # (expiry, version, key)
        self._clock = clock
        self._versions = itertools.count()                   # NEVER repeats - unlike a timestamp
        self.compactions = 0

    # ---- writes ----------------------------------------------------------
    def set(self, key: Any, value: Any, ttl: float) -> None:
        expiry = self._clock() + ttl
        version = next(self._versions)
        self._data[key] = (value, expiry, version)           # overwrite: the old version is now stale
        heapq.heappush(self._heap, (expiry, version, key))
        if len(self._heap) > 2 * len(self._data) + 32:
            self._compact()                                  # bound the accumulated garbage

    def delete(self, key: Any) -> bool:
        # The heap entry is left behind deliberately; it will be discarded when it surfaces.
        return self._data.pop(key, _MISSING) is not _MISSING

    # ---- reads -----------------------------------------------------------
    def get(self, key: Any, default: Any = _MISSING) -> Any:
        entry = self._data.get(key)
        if entry is None:
            return default
        value, expiry, _ = entry
        if expiry <= self._clock():
            del self._data[key]                              # expire on read, so get is always correct
            return default
        return value

    def __contains__(self, key: Any) -> bool:
        return self.get(key, _MISSING) is not _MISSING

    # ---- maintenance -----------------------------------------------------
    def clean_expired(self) -> int:
        now = self._clock()
        removed = 0
        while self._heap:
            expiry, version, key = self._heap[0]
            entry = self._data.get(key)
            if entry is None or entry[2] != version:
                heapq.heappop(self._heap)                    # STALE: deleted or overwritten
                continue
            if expiry <= now:
                heapq.heappop(self._heap)
                del self._data[key]                          # genuinely expired
                removed += 1
            else:
                break                                        # ordered: nothing behind this is due
        return removed

    def _compact(self) -> None:
        """Rebuild the heap from the live map, discarding accumulated garbage."""
        self._heap = [(exp, ver, k) for k, (_, exp, ver) in self._data.items()]
        heapq.heapify(self._heap)                            # O(n), cheaper than n pushes
        self.compactions += 1

    def size(self) -> int:
        self.clean_expired()
        return len(self._data)

    def next_expiry(self) -> Optional[float]:
        """When the soonest deadline falls - what a reaper thread would sleep until."""
        self.clean_expired()
        return self._heap[0][0] if self._heap else None

### Approach 3 — Thread-safe, with a background reaper

**Idea:** the follow-up asks how to reclaim entries proactively without blocking readers.

Two pieces, and the interesting part is how they interact:

- **A lock around every operation.** Each is O(1) or O(log n), so the critical section is tiny. Note that `get` **mutates** here — it deletes on expiry — so it is a writer, and a readers–writer lock would buy nothing. Same surprise as the [LRU cache](../10.%20LRU_Cache/10.%20LRU_Cache.ipynb), and worth naming.
- **A reaper that sleeps until the next deadline**, rather than polling on a fixed tick. `next_expiry()` tells it exactly how long to wait, so an idle table costs no CPU at all. It must be woken when a *sooner* deadline arrives — hence the `Event` — otherwise a 1-second TTL set while it sleeps for an hour would not be reclaimed until the hour was up.

**Time complexity:** unchanged, plus lock contention.

**Space complexity:** O(n).

In [ ]:
import threading


class ThreadSafeExpiringTable(ExpiringHashTable):
    """Adds a lock and a reaper that sleeps until the next deadline."""

    def __init__(self, clock: Callable[[], float] = time.monotonic) -> None:
        super().__init__(clock)
        self._lock = threading.RLock()           # RLock: size() calls clean_expired()
        self._wakeup = threading.Event()
        self._stop = threading.Event()
        self._reaper: Optional[threading.Thread] = None

    def set(self, key: Any, value: Any, ttl: float) -> None:
        with self._lock:
            super().set(key, value, ttl)
        self._wakeup.set()                       # a SOONER deadline may now exist

    def get(self, key: Any, default: Any = _MISSING) -> Any:
        with self._lock:                         # note: get MUTATES (it expires), so it is a writer
            return super().get(key, default)

    def delete(self, key: Any) -> bool:
        with self._lock:
            return super().delete(key)

    def clean_expired(self) -> int:
        with self._lock:
            return super().clean_expired()

    def size(self) -> int:
        with self._lock:
            return super().size()

    def start_reaper(self) -> None:
        def loop():
            while not self._stop.is_set():
                with self._lock:
                    super(ThreadSafeExpiringTable, self).clean_expired()
                    nxt = self._heap[0][0] if self._heap else None
                # Sleep exactly until the next deadline - no polling, no idle CPU.
                delay = max(0.0, nxt - self._clock()) if nxt is not None else 0.05
                self._wakeup.wait(timeout=min(delay, 0.05))
                self._wakeup.clear()

        self._reaper = threading.Thread(target=loop, daemon=True)
        self._reaper.start()

    def stop_reaper(self) -> None:
        self._stop.set()
        self._wakeup.set()
        if self._reaper is not None:
            self._reaper.join(timeout=2.0)

## Verification

A **fake clock** makes expiry deterministic — no sleeping, no flakiness. The checks cover the worked example, the overwrite and delete paths that create stale heap entries, and the identity bug the version counter fixes.

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor


class FakeClock:
    """A clock the tests drive by hand, so a one-hour TTL takes microseconds."""

    def __init__(self, t: float = 1000.0) -> None:
        self.t = t

    def __call__(self) -> float:
        return self.t

    def advance(self, dt: float) -> None:
        self.t += dt


# --- The worked example ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("a", 1, ttl=10)
t.set("b", 2, ttl=5)
assert t.get("a") == 1 and t.get("b") == 2
assert t.size() == 2

clk.advance(6)                                   # b's deadline has passed, a's has not
assert t.get("b", None) is None, "b must be invisible once its TTL has passed"
assert t.get("a") == 1, "a must still be live"
assert t.size() == 1

clk.advance(5)                                   # now a is past its deadline too
assert t.get("a", None) is None
assert t.size() == 0

# --- get expires even when clean_expired has never run ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("k", "v", ttl=1)
clk.advance(2)
assert t.get("k", None) is None, "a read must enforce the deadline itself"
assert "k" not in t._data, "and reclaim the entry on the spot"

# --- Exactly at the deadline: expiry is inclusive ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("k", "v", ttl=5)
clk.advance(4.999)
assert t.get("k") == "v", "still live just before the deadline"
clk.advance(0.001)
assert t.get("k", None) is None, "expiry <= now, so exactly at the deadline it is gone"

# --- A stored None is present, and distinct from absent ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("none", None, ttl=10)
assert t.get("none") is None and "none" in t
assert t.get("absent", "DEFAULT") == "DEFAULT"
assert "absent" not in t

# --- Overwriting extends the TTL, and leaves a stale heap entry ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("k", "first", ttl=5)
clk.advance(3)
t.set("k", "second", ttl=10)                     # a NEW deadline, 3 + 10 = 13
assert len(t._heap) == 2, "the old heap entry is left behind deliberately"
clk.advance(3)                                   # t = 6: past the FIRST deadline of 5
assert t.get("k") == "second", (
    "the stale heap entry must NOT expire the live value"
)
t.clean_expired()
assert t.get("k") == "second", "cleanup must recognise the stale entry and skip it"
assert len(t._heap) == 1, "...and discard it"
clk.advance(8)                                   # t = 14, past the second deadline of 13
assert t.get("k", None) is None

# --- delete leaves a stale heap entry that cleanup must ignore ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("k", "v", ttl=5)
assert t.delete("k") is True
assert t.delete("k") is False, "deleting twice is False, not an error"
assert len(t._heap) == 1, "the heap entry survives the delete"
clk.advance(10)
assert t.clean_expired() == 0, "a stale entry is discarded WITHOUT counting as an expiry"
assert len(t._heap) == 0

# --- THE identity bug: same expiry, different entry ---
# Set, expire via get, then set again at the SAME instant with the SAME ttl.
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("k", "old", ttl=5)
clk.advance(5)
assert t.get("k", None) is None                  # expired and removed from the map
t.set("k", "new", ttl=5)                         # expiry is now 1005 + 5 = 1010... and the
                                                 # STALE heap entry also has expiry 1005.
live_expiry = t._data["k"][1]
stale_expiry = min(e for e, _, _ in t._heap)
assert stale_expiry < live_expiry
t.clean_expired()
assert t.get("k") == "new", "the version counter keeps the live entry safe"

# A timestamp-only check would be fooled by a genuine collision:
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("k", "old", ttl=5)
old_expiry = t._data["k"][1]
t.delete("k")
t.set("k", "new", ttl=5)                         # SAME clock, SAME ttl => IDENTICAL expiry
assert t._data["k"][1] == old_expiry, "the two entries genuinely share an expiry timestamp"
assert t._data["k"][2] != t._heap[0][1], "...but NOT a version - which is what saves us"
clk.advance(10)
assert t.clean_expired() == 1, "exactly one real expiry, not a double-count"

# --- Cleanup stops at the first live entry ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
for i in range(10):
    t.set(f"k{i}", i, ttl=i + 1)                 # deadlines 1..10
clk.advance(3.5)
assert t.clean_expired() == 3, "k0, k1, k2 (deadlines 1, 2, 3) - and then it stops"
assert len(t._heap) == 7, "the loop must not walk past the first live entry"
assert t.size() == 7

# --- Degenerate TTLs ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
t.set("zero", "v", ttl=0)
assert t.get("zero", None) is None, "ttl=0 expires immediately (expiry <= now)"
t.set("neg", "v", ttl=-5)
assert t.get("neg", None) is None, "a negative ttl is already past"
t.set("long", "v", ttl=10 ** 9)
assert t.get("long") == "v"

# --- Garbage is bounded by compaction ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
for i in range(2000):
    t.set("hot", i, ttl=10 ** 6)                 # ONE key, overwritten 2000 times
assert len(t._data) == 1
assert t.compactions > 0, "the accumulated garbage must trigger compaction"
assert len(t._heap) <= 2 * len(t._data) + 32, f"heap garbage is unbounded: {len(t._heap)}"
assert t.get("hot") == 1999

# --- next_expiry ---
clk = FakeClock()
t = ExpiringHashTable(clock=clk)
assert t.next_expiry() is None, "an empty table has no next deadline"
t.set("a", 1, ttl=10)
t.set("b", 2, ttl=3)
assert t.next_expiry() == clk.t + 3, "the SOONEST deadline, not the most recent insert"

# --- Agreement with the naive implementation on randomised traffic ---
random.seed(109)
for _ in range(200):
    c1, c2 = FakeClock(), FakeClock()
    fast, naive = ExpiringHashTable(clock=c1), NaiveExpiringTable(clock=c2)
    for _ in range(random.randint(1, 60)):
        k = f"k{random.randrange(8)}"
        r = random.random()
        if r < 0.5:
            v, ttl = random.randrange(100), random.uniform(0.5, 5)
            fast.set(k, v, ttl); naive.set(k, v, ttl)
        elif r < 0.7:
            assert fast.get(k, None) == naive.get(k, None), k
        elif r < 0.85:
            assert fast.delete(k) == naive.delete(k)
        else:
            dt = random.uniform(0, 2)
            c1.advance(dt); c2.advance(dt)
    c1.advance(1); c2.advance(1)
    assert fast.size() == naive.size()
    for i in range(8):
        assert fast.get(f"k{i}", None) == naive.get(f"k{i}", None)

# --- Thread safety, and the reaper ---
ts = ThreadSafeExpiringTable()                   # a REAL monotonic clock here


def hammer(worker):
    for i in range(300):
        ts.set(f"w{worker}-{i}", i, ttl=60)
        ts.get(f"w{worker}-{i}", None)


with ThreadPoolExecutor(max_workers=8) as ex:
    list(ex.map(hammer, range(8)))
assert ts.size() == 8 * 300, f"concurrent writes lost entries: {ts.size()}"

reaped = ThreadSafeExpiringTable()
reaped.start_reaper()
try:
    for i in range(50):
        reaped.set(f"short{i}", i, ttl=0.05)     # expires almost immediately
    reaped.set("long", "stays", ttl=60)
    deadline = time.monotonic() + 3.0
    while reaped.size() > 1 and time.monotonic() < deadline:
        time.sleep(0.02)
    assert reaped.size() == 1, "the reaper must reclaim expired entries on its own"
    assert reaped.get("long") == "stays", "and leave the live one alone"
finally:
    reaped.stop_reaper()

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **A background reaper.** Implemented above. The design choice worth defending is **sleeping until the next deadline** rather than polling on a fixed tick: `next_expiry()` says exactly how long to wait, so an idle table burns no CPU. The catch is that a *sooner* deadline can arrive while the reaper sleeps, so `set` must wake it — hence the `Event`. Without that, a one-second TTL created just after the reaper settled in for an hour would survive the hour.
- **Lazy versus eager expiry.** Redis does **both**, and the reason is instructive: lazy expiry (on read) is free but never reclaims keys nobody touches again; a reaper reclaims them but costs CPU. Redis samples a random subset of keys periodically rather than scanning, precisely to avoid the O(n) walk. Naming that combination is a strong answer.
- **Persistence.** A write-ahead log of `set`/`delete` operations, replayed on boot — the machinery from [`3. Persistent_Append_Only_Log`](../3.%20Persistent_Append_Only_Log/3.%20Persistent_Append_Only_Log.ipynb). The wrinkle unique to TTLs: **do you persist the absolute deadline or the remaining lifetime?** An absolute timestamp means everything expires at once if the process was down for an hour; a remaining duration means a key can outlive its intended deadline. Redis stores absolute times and accepts the first behaviour. Either is defensible; silently picking one is not.
- **Capping memory.** TTL bounds *time*, not *size* — a burst of long-TTL keys can still exhaust memory. Layering a max-size policy on top means choosing what to evict when full, and the natural answer is a second index: LRU order for "least useful", or the existing expiry heap for "soonest to die anyway". That composition is exactly Redis's `volatile-lru` versus `volatile-ttl` eviction policies.
- **`ttl <= 0`.** Asserted above: the entry is written and immediately invisible, because `expiry <= now`. That falls out of the comparison rather than being special-cased, which is the behaviour to *state* rather than discover. Rejecting it with a `ValueError` is equally defensible — what matters is that the semantics are decided, not accidental.
- **Batch `get_many`.** Take the lock once, `clean_expired` once, then read all the keys — which turns k lock acquisitions into one. The subtle benefit is **atomicity**: the whole batch is evaluated against a single instant, so you cannot see key A alive and key B expired when both share a deadline.

## Empirical complexity check

Compare **scanning for expired entries** with the **heap**, on a table where only a handful of entries are actually due. This is the realistic shape: most keys are nowhere near their deadline.

| Growth when the table doubles | What it means |
|---|---|
| ~2x | linear — every entry is inspected, however far from expiring |
| ~1x | proportional to what is *reclaimed*, not to what is stored |

That is the whole argument for the heap: cleanup cost should track the number of **expired** entries, not the number of entries.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

CLEANUPS = 200


def make_table(n):
    """n entries, only ~20 of which are due to expire soon."""
    clk = FakeClock()
    fast, naive = ExpiringHashTable(clock=clk), NaiveExpiringTable(clock=clk)
    for i in range(n):
        ttl = 0.5 if i < 20 else 10 ** 6         # 20 short-lived, the rest effectively immortal
        fast.set(f"k{i}", i, ttl)
        naive.set(f"k{i}", i, ttl)
    clk.advance(1)                               # the 20 are now due
    return (fast, naive, clk)


def run_scan(fast, naive, clk):
    for _ in range(CLEANUPS):
        naive.clean_expired()                    # O(n): inspects EVERY entry


def run_heap(fast, naive, clk):
    for _ in range(CLEANUPS):
        fast.clean_expired()                     # O(k log n): only what is actually due


benchmark(
    {"Approach 1 - scan every entry O(n)": run_scan,
     "Approach 2 - min-heap O(k log n)": run_heap},
    make_table,
    sizes=[2000, 4000, 8000, 16000],
    repeats=1,
)

## Patterns learned

- **Two questions, two structures, one set of entries.** A hash map answers "what is this key worth?"; a min-heap answers "what dies next?". Neither can do the other's job, and composing them is the same move as the [LRU cache](../10.%20LRU_Cache/10.%20LRU_Cache.ipynb) with a different second index.
- **A heap cannot be edited in place, so use lazy deletion.** Leave the obsolete entry, detect it when it surfaces, and rebuild once the garbage outgrows the live set. The map stays the source of truth; the heap is only a hint about when to look.
- **Identify an entry by a version, never by a value that can repeat.** Timestamps collide. A monotonic counter cannot — and it doubles as a deterministic heap tie-break, so keys of mixed types are never compared.
- **Read the monotonic clock.** Wall time can jump backwards; TTLs computed from it silently extend. This is correctness, not polish.
- **Inject the clock.** A fake clock turns an hour-long TTL into a microsecond test, and removes every source of flakiness. Same discipline as injecting `sleep` in [Retry Strategy](../18.%20Retry_Strategy/18.%20Retry_Strategy.ipynb).
- **An ordered structure lets you stop early.** `clean_expired` breaks at the first live entry because everything behind it is later. Cleanup then costs what it *reclaims*, not what it *stores*.
- **`get` that expires is a writer.** It mutates, so a readers–writer lock buys nothing — the same counter-intuitive property as an LRU cache's `get`.
- **Bound the garbage.** Lazy deletion trades promptness for speed, and without a compaction threshold that trade becomes an unbounded leak on a hot key.